In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [ ]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and start a local model:
   ```bash
   ollama run llama3
   ```

   This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test that the local server is running:
   ```bash
   curl http://localhost:11434
   ```

   You should get a response like:
   ```json
   {"models": [...]}  
   ```

If you want to use it from Python, install the client with:

```bash
pip install ollama
```

Then you can call it with:

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```


In [ ]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any FAQ entry about **Olama/Ollama** specifically.

The closest relevant guidance is that you **can run the course locally** instead of using Codespaces, as long as you’re comfortable setting up the needed tools like:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

If you run locally, make sure to **document your setup** and keep the environment **reproducible**.

If you meant **Ollama** specifically, the context here doesn’t include instructions for it.


In [10]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Absolutely — if the course is still open, you can usually join it.\n\nThe quickest way to confirm is to check:\n- the course page or enrollment link\n- the start date and whether registration is still open\n- any prerequisites or eligibility requirements\n\nIf you want, I can help you draft a short message to the instructor or support team asking to join.'

In [11]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [12]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [13]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [9]:
len(response.output)

1

In [12]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it started? discovered the course can I join it late enrollment"}', call_id='call_tVKdk8QKBB7mnDLmMTo78sxk', name='search', type='function_call', id='fc_0d87336b00bb7331006a3608b78180819db0b29be59deafe18', namespace=None, status='completed')]

In [14]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll register can I join the course after it started"}', call_id='call_7zK4jFVuQT4PMGwVb42Wb4Xw', name='search', type='function_call', id='fc_06ea0835c1184d61006a360e21cc50819f93dbb0c24dbae24e', namespace=None, status='completed')

In [15]:
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered late enroll register can I join the course after it started'}

In [15]:
call.name

'search'

In [17]:
results = search(**args)
results 

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I sta

In [19]:
result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course

In [20]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [24]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll register can I join the course after it started"}', call_id='call_7zK4jFVuQT4PMGwVb42Wb4Xw', name='search', type='function_call', id='fc_06ea0835c1184d61006a360e21cc50819f93dbb0c24dbae24e', namespace=None, status='completed')]

In [21]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll register can I join the course after it started"}', call_id='call_7zK4jFVuQT4PMGwVb42Wb4Xw', name='search', type='function_call', id='fc_06ea0835c1184d61006a360e21cc50819f93dbb0c24dbae24e', namespace=None, status='completed')

In [23]:
messages.append(call)

In [ ]:
messages

In [ ]:
# messages.extend(response.output)

# messages.append({
#  "type": "function_call_output",
#  "call_id": call.call_id,
#  "output": result_json,
# })

In [25]:
messages.append(function_call_output)

In [26]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll register can I join the course after it started"}', call_id='call_7zK4jFVuQT4PMGwVb42Wb4Xw', name='search', type='function_call', id='fc_06ea0835c1184d61006a360e21cc50819f93dbb0c24dbae24e', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_7zK4jFVuQT4PMGwVb42Wb4Xw',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode